In [ ]:
import sys
import os
root_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.insert(0, str(root_dir))   


############################################## Dataset
# from datasets.rescuenet import Dataset
# from datasets.floodnet import Dataset
from datasets.cracks import Dataset
######################################################


from tqdm import tqdm 

from PIL import Image, ImageOps
import numpy as np


In [9]:
# #---------------------------------------- rescuenet
# Dataset.stats_from_yaml('rescuenet.yaml')
# dataset = Dataset(split='test', scale=3000)

# #---------------------------------------- floodnet
# Dataset.stats_from_yaml('floodnet.yaml')
# dataset = Dataset(split='test', scale=704)

#---------------------------------------- public cracks
subfolders = [
              'ConcreteCrack', 
              #'SyntheticCracks',
              'Stone331',
              'CrackTree260', 
              'DeepCrack', 
              'CrackLS315',         
              'CrackForest',             
              'CRKWH100', 
              ]  

public = "public-cracks"
Dataset.stats_from_yaml('cracks-public.yaml')
dataset = Dataset(split='test', root_path=public, subfolders=subfolders)


# # --------------------------------------- photos
# photos ="Dataset_2D_Train_Undersampled_No_CastelNuovo_AreaGrande_CentroItalia"
# Dataset.stats_from_yaml('cracks-photos.yaml')
# dataset = Dataset(split='test', root_path=photos)


# #---------------------------------------- textures
# textures =  "Dataset_Textures_Train_Undersampled_2.0"
# Dataset.stats_from_yaml('cracks-textures.yaml')
# dataset = Dataset(split='test', root_path=textures)


# #---------------------------------------------- textures + photos : Crack-concat
# Dataset.stats_from_yaml('cracks-concat.yaml')
# concat =  "Cracks-concat"
# dataset = Dataset(split='train', root_path=concat)



In [10]:
path = dataset._get_path(1)[0]
print(f"Loading image: {path}")
img = Image.open(path)
img_array = np.array(img)

print("Image Pixel Value Range:")
print(f"Min: {img_array.min()}, Max: {img_array.max()}, Data Type: {img_array.dtype}")

Loading image: /media/mauro/Data/Datasets/public-cracks/ConcreteCrack/test/imgs/028.jpg
Image Pixel Value Range:
Min: 50, Max: 255, Data Type: uint8


In [11]:
paths = [dataset._get_path(i)[0] for i in range(len(dataset))]

In [12]:

# Variables to store the sum of RGB values
r_sum = 0
g_sum = 0
b_sum = 0

# Variable to store total number of pixels
total_pixels = 0

# First pass: Calculate mean RGB values
for path in tqdm(paths, desc="Calculating Mean"):
    # Open image and convert to a numpy array
    img = Image.open(path).convert('RGB')  # Ensure the image is in RGB format
    img_array = np.array(img)  # Convert to numpy array (shape: H, W, 3)

    # Get the height, width, and channels of the image
    height, width, _ = img_array.shape
    
    # Sum the values for each channel
    r_sum += img_array[:, :, 0].sum()  # Red channel sum
    g_sum += img_array[:, :, 1].sum()  # Green channel sum
    b_sum += img_array[:, :, 2].sum()  # Blue channel sum

    # Update the total number of pixels
    total_pixels += height * width  # Number of pixels in the current image


# Calculate the mean for each channel
r_mean = r_sum / total_pixels
g_mean = g_sum / total_pixels
b_mean = b_sum / total_pixels

print(f"Mean RGB values before preprocessing: R: {r_mean}, G: {g_mean}, B: {b_mean}")


Calculating Mean: 100%|██████████| 322/322 [00:00<00:00, 429.22it/s]

Mean RGB values before preprocessing: R: 150.16928023581178, G: 148.4208561204235, B: 145.93543536026286


In [13]:

# Second pass: Calculate the standard deviation
r_var_sum = 0
g_var_sum = 0
b_var_sum = 0

for path in tqdm(paths, desc="Calculating Standard Deviation"):
    # Open image and convert to a numpy array
    img = Image.open(path).convert('RGB')
    img_array = np.array(img)

    # Calculate variance for each channel by summing the squared differences from the mean
    r_var_sum += ((img_array[:, :, 0] - r_mean) ** 2).sum()
    g_var_sum += ((img_array[:, :, 1] - g_mean) ** 2).sum()
    b_var_sum += ((img_array[:, :, 2] - b_mean) ** 2).sum()


# Calculate the variance by dividing by the total number of pixels
r_var = r_var_sum / total_pixels
g_var = g_var_sum / total_pixels
b_var = b_var_sum / total_pixels

# Calculate the standard deviation by taking the square root of the variance
r_std = np.sqrt(r_var)
g_std = np.sqrt(g_var)
b_std = np.sqrt(b_var)

# Output the mean and standard deviation RGB values
print(f"Standard Deviation RGB values before preprocessing: R: {r_std}, G: {g_std}, B: {b_std}")


Calculating Standard Deviation: 100%|██████████| 322/322 [00:01<00:00, 291.57it/s]

Standard Deviation RGB values before preprocessing: R: 50.24395969544343, G: 49.19092032629675, B: 48.16331308891341
